# Navier–Stokes experiments

This notebook is the single entry point for the shock-flow weighting and gradient-diagnostic experiments. Edit only the **Run controls** cell and run all cells. Each execution launches one W&B run.

The PINN is trained physics-only with three losses: initial conditions, periodic boundary conditions, and the compressible Navier–Stokes residual. No interior solution observations are used for training.

For fair comparisons, change one control at a time and use matched seeds:

- **Loss weighting:** choose `BALANCER = 'softadapt'` or `'gradient_annealing'`; for SoftAdapt, compare `SOFTADAPT_BETA = 0.0` (uniform) with a positive beta.
- **Gradient diagnostics:** gradient norms, cosine similarities, and per-loss gradient histograms are always logged to W&B.

In [ ]:
%pip install -q wandb accelerate

## Kaggle setup

In [ ]:
import os

base_path = '/kaggle/working'
project_name = 'weight-vs-loss-convergence-pinn'
project_root = os.path.join(base_path, project_name)

if not os.path.exists(project_root):
    os.chdir(base_path)
    !git clone https://github.com/nthday-jpg/weight-vs-loss-convergence-pinn.git
else:
    os.chdir(project_root)
    !git pull

os.environ['PYTHONPATH'] = project_root
%cd {project_root}

from kaggle_secrets import UserSecretsClient
wandb_api_key = UserSecretsClient().get_secret('WANDB_KEY').strip()
if len(wandb_api_key) < 40:
    raise ValueError(
        'Kaggle secret WANDB_KEY is not a valid W&B API key. '
        'Replace it with the key from https://wandb.ai/authorize.'
    )
os.environ['WANDB_API_KEY'] = wandb_api_key

## Run controls

Training is always physics-only on the shock-forming case. The reference data supplies the initial condition, physical constants, and held-out evaluation labels.

In [ ]:
BALANCER = 'softadapt'       # softadapt or gradient_annealing
SOFTADAPT_BETA = 0.0           # 0.0 is uniform; use 0.1 for adaptation
SEED = 42

## Prepare the benchmark data

In [ ]:
import subprocess
import sys

if not isinstance(SOFTADAPT_BETA, (int, float)) or SOFTADAPT_BETA < 0:
    raise ValueError('SOFTADAPT_BETA must be a nonnegative number')

base_name = 'navier_stokes_shock'
data_file = os.path.join(project_root, 'navier-stokes', 'data', base_name + '.pt')

subprocess.run([
    sys.executable, 'navier-stokes/data/gen_data.py',
    '--save_dir', 'navier-stokes/data',
    '--base_name', base_name,
], check=True)

## Build and review the training configuration

In [ ]:
import json

config = {
    'problem': 'navier_stokes',
    'layers': [2, 64, 64, 64, 3],
    'learning_rate': 1e-3,
    'num_steps': 50_000,
    'adapt_every_steps': 100,
    'log_every_steps': 100,
    'batch_size': 8192,
    'seed': SEED,
    'train_time_fraction': 0.7,
    'validation_time_fraction': 0.85,
    'wandb_project': 'loss-weight-convergence_navier',
    'balancer': BALANCER,
    'softadapt_params': {
        'beta': SOFTADAPT_BETA,
        'history_length': 5,
        'normalized': True,
        'loss_weighted': False,
    },
}

config['wandb_run_name'] = (
    f"shock-{config['balancer']}"
)

os.environ['PINN_CONFIG'] = json.dumps(config)
os.environ['PINN_DATA_FILE'] = data_file

print(json.dumps(config, indent=2))
print(f'\nData: {data_file}')

## Train

A single process keeps adaptive-balancer state deterministic and synchronized.

In [ ]:
!accelerate launch {project_root}/train.py

In [ ]:
from IPython.display import Image, display
from pathlib import Path

from share.artifacts import comparison_plot_path

plot_path = comparison_plot_path(
    config['wandb_run_name'], Path(project_root) / 'results'
)
if plot_path.exists():
    display(Image(filename=str(plot_path)))
else:
    print(f'Comparison plot not found at {plot_path}. Check the training output above.')
print('Convergence tables and diagnostic plots are available in the W&B run.')